# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [28]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import os
import asyncio

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    if value:
        return value
    raise ValueError(f"{key} not found")

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

high_effort_model = "gpt-5.6-sol"
default_model = "gpt-5.6-luna"
low_effort_model = "gpt-5.6-terra"

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)

print("Models loaded ✔")

base_system_instructions = '''
You are a meal planning assistant who helps people plan their meals for the week.
You are an expert on simple meals that require minimal amounts of preparation, reheat well, and are delicious.
'''

API keys loaded ✔
Models loaded ✔


In [ ]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def send_push_notification(message: str):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## Memory
Start with holding things in memory for now and then I can move to persistence later.
Use functions so it's abstracted and easy to swap later.

## User Preferences 
Collects dietary restrictions, likes/dislikes, meals that they're tired of, and what kind of cooking equipment they have. 

In [ ]:
class UserPreferences(BaseModel):
    dietary_restrictions: list[str] = Field(default_factory=list, description="A list of any dietary restrictions the user has that need to be considered for the meal plan.", examples=["Gluten-free", "vegetarian", "dairy-free"])
    likes: list[str] = Field(default_factory=list, description="A list of foods user's favorite foods.")
    dislikes: list[str] = Field(default_factory=list, description="A list of foods that the user does not like.")
    nutritional_goals: str = Field(default="", description="A description of the nutritional goals that the user aims to achieve with this meal plan.", examples=["Increase protein intake, lose weight, lower cholesterol"])
    meals_to_avoid_this_time: list[str] = Field(default_factory=list, description="Specific foods that the user would prefer to avoid this plan.")
    notes: str = Field(default="", description="A paragraph of notes about any preferences that don't apply to one of the other fields.", examples=["Half of my meals should be meatless."])

class UserPreferencesReview(BaseModel):
    preferences: UserPreferences = Field(description="The user's updated preferences.")
    user_has_confirmed_that_preferences_are_correct: bool = Field(description="True only when the user has reviewed their preferences, confirmed that they are correct, and that they don't want to modify it any further.")
    follow_up_response: str = Field(description="Your polite response to the user's message confirming that you understand their request. If the user has not yet confirmed that the preferences are complete, ask them if there is anything else.")

saved_user_preferences = UserPreferences()

def set_user_preferences(update: UserPreferences):
    global saved_user_preferences
    saved_user_preferences = update

def get_user_preferences() -> UserPreferences:
    return saved_user_preferences

user_preferences_system_instructions = f'''
{base_system_instructions}
'''

user_preferences_prompter = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
)

user_preferences_reviewer = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
    output_type=UserPreferencesReview,
)

async def present_known_user_preferences() -> str:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    Present it to the user in an easily reviewable format.

    You need the correct preferences to make sure that the meal plan works for the user. It's essential for you to do your job correctly.
    Explain the importance to the user.
    Then ask the user to either confirm that everything looks correct or make changes.
    '''

    return (await Runner.run(user_preferences_prompter, prompt)).final_output

async def update_user_preferences(user_response: str) -> UserPreferencesReview:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    You presented this information to the user.

    This was the user's response:
    {user_response}

    If the user has not yet confirmed that the preferences are correct and complete, prompt them to finish it in the follow_up_response field.
    '''

    return (await Runner.run(user_preferences_reviewer, prompt)).final_output


In [57]:
set_user_preferences(UserPreferences(dietary_restrictions=["egg allergy"],nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless"))

In [58]:
with trace("User Preferences Presenter Test"):
    print(await present_known_user_preferences())

## Your Meal Plan Preferences

- **Dietary restriction:** Egg allergy  
- **Goals:** Lose weight; high protein  
- **Preference:** Half of meals should be meatless  
- **Likes:** None specified  
- **Dislikes:** None specified  
- **Meals to avoid this time:** None  
- **Additional notes:** None  

These details help ensure the plan is safe, supports your goals, and includes the right balance of meatless meals.

Please confirm that everything is correct, or tell me what you’d like to change.


In [60]:
initial_prompt ='''
## Your Meal Plan Preferences

I don't see any preferences provided yet.

Please share your preferences, such as:

- Dietary restrictions or allergies
- Foods you like or dislike
- Number of meals and servings
- Cooking time and equipment
- Budget
- Preferred cuisines
- Meals suitable for reheating

Once provided, I'll format them clearly for you to review and confirm.
'''

with trace("User Preferences Review Test"):
    print(f"{initial_prompt}\n\n")
    first_user_response = "I want half of my meals to be meatless. I want my food to be relatively healthy too so I can try to lose wait as well."
    print(f"{first_user_response}\n\n")
    first_update = await update_user_preferences(
        first_user_response,
    )
    print(f"{first_update}\n\n")
    

    second_user_response = "That looks correct, thank you."
    print(f"{second_user_response}\n\n")
    second_update = await update_user_preferences(
        second_user_response,
    )
    print(f"{second_update}\n\n")



## Your Meal Plan Preferences

I don't see any preferences provided yet.

Please share your preferences, such as:

- Dietary restrictions or allergies
- Foods you like or dislike
- Number of meals and servings
- Cooking time and equipment
- Budget
- Preferred cuisines
- Meals suitable for reheating

Once provided, I'll format them clearly for you to review and confirm.



I want half of my meals to be meatless. I want my food to be relatively healthy too so I can try to lose wait as well.


preferences=UserPreferences(dietary_restrictions=['egg allergy'], likes=[], dislikes=[], nutritional_goals='Lose weight; prioritize high-protein, relatively healthy meals.', meals_to_avoid_this_time=[], notes='Half of the meals should be meatless.') user_has_confirmed_that_preferences_are_correct=True response="Thanks—I've confirmed your preferences: egg-free, high-protein meals focused on weight loss, with half of the meals meatless."


That looks correct, thank you.


preferences=UserPreferences(

## Meal Brainstorming Agent

An agent that generates a bunch of meal ideas.

TODO: Come up with ideas on how make it change the set of recipes. 
* It can have a sentence to specify the month on the first line. That would make the food seasonal too.
* I can have have LLMs write variations of the instructions and randomly pick one.
* I can randomly assign the model that generates the ideas

In [ ]:
class PreparedDish(BaseModel):
    name: str = Field(description="The short name of a dish.")
    description: str = Field(description="A 1-2 sentence description of the dish.")
    special_diet_labels: str = Field(description="A list of any dietary restrictions that this meal satisfies", examples="vegetarian, gluten-free")
    category: str = Field(description="The category of food.", examples = "Italian, Chinese")


class PreparedDishes(BaseModel):
    dishes: list[PreparedDish] = Field(description="A list of ideas for prepared dishes.")

class MealPlanIdeas:
    entree_ideas: list[PreparedDish]
    side_ideas: list[PreparedDish] 

    def __init__(self, entree_ideas: list[PreparedDish], side_ideas: list[PreparedDish]) -> None:
        self.entree_ideas = entree_ideas
        self.side_ideas = side_ideas

    def __str__(self):
        return f"entrees: {self.entree_ideas}, sides: {self.side_ideas}"


system_instructions = f'''
{base_system_instructions}
'''

meal_brainstorming_agent = Agent(
    name="Meal Brainstormer",
    instructions=system_instructions,
    model=default_model,
    output_type=PreparedDishes,
)

async def generate_meal_ideas(prompt: str) -> PreparedDishes:
    return (await Runner.run(meal_brainstorming_agent, prompt)).final_output.dishes


async def create_meal_plan_brainstorm(number_of_meals: int, preferences: UserPreferences) -> MealPlanIdeas:
    user_preferences_prompt = f'''
    Here is the user's preferences:
    {preferences}
    '''
    
    meal_idea_multiple = 4 # generate extra ideas in case we need to drop some of them
    entrees, sides = await asyncio.gather(
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different entrees. {user_preferences_prompt}"),
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different sides. {user_preferences_prompt}")
    )
    return MealPlanIdeas(
        entree_ideas = entrees, 
        side_ideas = sides,
    )

NameError: name 'UserPreferences' is not defined

In [ ]:
with trace("Meal Brainstorming Test"):
    preferences = UserPreferences(nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless")
    unfiltered_meal_plan = create_meal_plan_brainstorm(2, preferences)
    print(unfiltered_meal_plan)

## Meal Validation Agent
Filters brainstorm ideas that don't conform to likes, dislikes, and user preferences.
Verifies that it's not too repetitive with the meals from last week.

## Pairing Agent
Picks meals and attempts to pair it with similar side based on category.

## User Feedback Agent
Asks user to approve the selected meals. Can make changes to a meal or select a different one from the brainstorming list. If all else fails, can escape back and restart with a different model.

## Recipe Agent
Writes recipes for the selected meals

## Recipe Validation Agent
Makes sure that the recipes conform to the user's dietary restrictions and can be made with the user's equipment.

## Shopping List Agent
Collects all of the ingredients from the recipes above

## Shopping List Formatting Agent
Combines identical ingredients and groups them by section of the super market

## Meal Plan Presentation Agent
Turns everything into pretty markdown:
* High level plan
* Shopping List
* Recipes

## Orchestration
Ties everything together and runs it in a UI.

### Basic Sequential Iteration
Goes through steps one by one.

In [ ]:
with trace("Meal Planning"):
    unfiltered_meal_plan = create_meal_plan_brainstorm(2, get_user_preferences())
    print(unfiltered_meal_plan)

### LLM Orchestration
Use tools and subagents to allow better flexibility like moving backwards to change preferences or meal choices